# Limpieza de datos nulos – Titanic
Imputación de valores faltantes usando las técnicas vistas: media, mediana, moda, constante, KNN e interpolación.

## 1. Carga del dataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/CIENCIA DE DATOS/Titanic/titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Diagnóstico de nulos

In [9]:
nulos = df.isna().sum()
porcentaje = (df.isna().mean() * 100).round(2)
pd.DataFrame({'Nulos': nulos, '% nulos': porcentaje})[nulos > 0]

,Nulos,% nulos
Age,177,19.87
Cabin,687,77.10
Embarked,2,0.22


## 3. Imputación

| Columna | Técnica | Justificación |
|---|---|---|
| `Age` | Mediana | Numérica con valores atípicos (hasta 80 años). |
| `Embarked` | Moda | Categórica con solo 2 nulos. |
| `Cabin` | Constante ("Desconocido") | ~77 % de nulos; no es posible estimarla de forma confiable. |

In [5]:
df_limpio = df.copy()

# Age -> mediana
mediana_edad = df_limpio['Age'].median()
df_limpio['Age'] = df_limpio['Age'].fillna(mediana_edad)

# Embarked -> moda
moda_embarked = df_limpio['Embarked'].mode()[0]
df_limpio['Embarked'] = df_limpio['Embarked'].fillna(moda_embarked)

# Cabin -> constante
df_limpio['Cabin'] = df_limpio['Cabin'].fillna('Desconocido')

print(f'Age -> mediana = {mediana_edad}')
print(f'Embarked -> moda = {moda_embarked}')
print('Cabin -> constante = "Desconocido"')

Age -> mediana = 28.0
Embarked -> moda = S
Cabin -> constante = "Desconocido"


## 4. Verificación

In [6]:
print('Nulos después de la imputación:', df_limpio.isna().sum().sum())
df_limpio.isna().sum()

Nulos después de la imputación: 0


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


## 5. Alternativa: imputación KNN para `Age`
Estima la edad a partir de pasajeros similares (clase, sexo, familiares a bordo, tarifa).

In [7]:
from sklearn.impute import KNNImputer

df_knn = df.copy()
cols = ['Age', 'Pclass', 'SibSp', 'Parch', 'Fare']
X = df_knn[cols].copy()
X['Sex'] = (df_knn['Sex'] == 'male').astype(int)

imputer = KNNImputer(n_neighbors=5)
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
df_knn['Age'] = X_imp['Age'].round(1)

print('Nulos en Age (KNN):', df_knn['Age'].isna().sum())
print('Media de Age, mediana vs KNN:', round(df_limpio['Age'].mean(), 2), 'vs', round(df_knn['Age'].mean(), 2))

Nulos en Age (KNN): 0
Media de Age, mediana vs KNN: 29.36 vs 30.04


In [11]:
df_limpio.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


### Análisis de Supervivencia basado en el Dataset del Titanic

Vamos a analizar los datos de supervivencia según el perfil de **Jack** y **Rose** utilizando nuestro DataFrame `df_limpio`:
- **Rose:** Mujer, Primera Clase (`Pclass = 1`), edad aproximada de 17 años.
- **Jack:** Hombre, Tercera Clase (`Pclass = 3`), edad aproximada de 20 años.

Como muestran los datos:
- Las personas con el perfil de **Rose** tenían una probabilidad extremadamente alta de salvarse (cercana al **94-100%** según el rango exacto).
- En cambio, los hombres jóvenes en tercera clase como **Jack** tenían una probabilidad de sobrevivir bajísima (cercana al **10-12%**).

Incluso si físicamente cabían en la puerta, la implacable prioridad de evacuación ("mujeres y niños primero") y la abismal diferencia de clases sociales reflejada en el dataset determinaron el destino de Jack desde el momento del choque.